In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    from ipywidgets import interact, IntSlider
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    from ipywidgets import interact, IntSlider

# Ensure paths are correct for your local structure
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. VISUALIZATION FUNCTION
# ==========================================
def plot_densoinn_topology(X_raw, nodes, edges, macro_clusters, batch_name="Current"):
    """
    Renders a clean, paper-style graph topology visualization for DenSOINN.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # 1. Background: Draw the raw data points
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=15, alpha=0.6, label='Raw Data', zorder=1)

    # Map unique macro-cluster IDs to distinct colors
    unique_mcs = sorted(list(set(macro_clusters.values())))
    cmap = plt.get_cmap('tab20') 
    color_map = {mc_id: cmap(i % 20) for i, mc_id in enumerate(unique_mcs)}

    drawn_edges = set()

    # 2. Midground: Draw Graph Edges (Topological Connections)
    for src_id, targets in edges.items():
        if src_id not in nodes: continue
        c1 = nodes[src_id]['centroid']
        
        # Edges take the color of their source node's macro-cluster
        mc_id = macro_clusters.get(src_id, -1)
        color = color_map.get(mc_id, 'grey') if mc_id != -1 else 'black'

        for tgt_id in targets:
            if tgt_id not in nodes: continue
            
            edge_tuple = tuple(sorted((src_id, tgt_id)))
            if edge_tuple not in drawn_edges:
                c2 = nodes[tgt_id]['centroid']
                ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color=color, linewidth=2.0, alpha=0.8, zorder=2)
                drawn_edges.add(edge_tuple)

    # 3. Foreground: Draw Neural Nodes
    for node_id, node_data in nodes.items():
        centroid = node_data['centroid']
        mc_id = macro_clusters.get(node_id, -1)
        color = color_map.get(mc_id, 'grey') if mc_id != -1 else 'black'
        
        # Draw node as an 'X'
        ax.scatter(centroid[0], centroid[1], color=color, marker='X', edgecolor='black', linewidth=0.5, s=80, zorder=3)

    # Live Metrics Sidebar
    num_macro_clusters = len([m for m in unique_mcs if m != -1])
    num_nodes = len(nodes)
    
    stats_text = (
        f"DenSOINN Live Metrics\n"
        f"---------------------\n"
        f"Macro-Clusters: {num_macro_clusters}\n"
        f"Neural Nodes: {num_nodes}"
    )
    
    bbox_props = dict(boxstyle='round,pad=0.6', facecolor='whitesmoke', alpha=0.9, edgecolor='lightgrey')
    ax.text(1.05, 0.5, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='center', bbox=bbox_props, fontfamily='monospace')

    ax.set_title(f"DenSOINN Topology - {batch_name}", fontsize=14, fontweight='bold')
    ax.set_aspect('equal', adjustable='datalim') 
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    # Custom Legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Raw Data', markerfacecolor='gainsboro', markersize=8),
        Line2D([0], [0], marker='X', color='w', label='Neural Node', markerfacecolor=cmap(0), markeredgecolor='black', markersize=10),
        Line2D([0], [0], color=cmap(0), lw=2.0, label='Topological Edge'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.35, 1), frameon=False)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 2. SETUP PARAMETERS & LOAD REAL DATA
# ==========================================
dataset_name = 'complex9'
batch_size = 300          

# DenSOINN Hyperparameters (Tuned for complex9 based on your earlier discussions)
gamma = 0.25           # Edge decay speed
stream_speed = 200     # Denoising frequency
epsilon = 0.5          # Denoising intensity
max_nodes = 1000       # Network capacity limit
alpha = 5.0            # Offline peak sensitivity multiplier
intrinsic_dim = 2      # Dimensionality for density estimation

print(f"Loading '{dataset_name}' data...")
# DenSOINN's self-adaptive metric handles unnormalized data natively, but we follow the runner's scaling default
X, y = load_data(dataset_name, scale=True) 

densoinn = DenSOINN(
    gamma=gamma,
    stream_speed=stream_speed,
    epsilon=epsilon,
    max_nodes=max_nodes,
    alpha=alpha,
    intrinsic_dim=intrinsic_dim
)

num_batches = len(X) // batch_size
stream_history = []
all_X = []

print(f"Processing {num_batches} batches...")

# ==========================================
# 3. RUN ALGORITHM & CAPTURE STATE
# ==========================================
for i in range(num_batches + 1):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(X))
    batch = X[start_idx:end_idx]

    if len(batch) == 0:
        break

    timestamps = np.arange(start_idx, end_idx) 
    
    # Run Layer 1 (Online Phase)
    densoinn.learn(batch, timestamps)
    all_X.extend(batch)

    # --- STATE EXTRACTION ---
    # 1. Deep copy the Neural Nodes
    current_nodes = {}
    for n_id, node in densoinn.online_phase.network.nodes.items():
        current_nodes[n_id] = {'centroid': node.w.copy()}

    # 2. Extract the Topological Edges (Weakly connected)
    current_edges = {}
    for src, tgts in densoinn.online_phase.network.edges.items():
        current_edges[src] = list(tgts.keys())

    # 3. Trigger Layer 2 (Offline Phase) to get live macro-clusters
    summaries = densoinn.online_phase.get_summary()
    current_macro = densoinn.offline_phase.predict_summaries(summaries)

    # Save the frame
    stream_history.append({
        'batch_name': f'Batch {i} (t={timestamps[-1]})',
        'X_raw': np.array(all_X), 
        'nodes': current_nodes,
        'edges': current_edges,
        'macro_clusters': current_macro,
    })

print("Stream processing complete! Ready for visualization.")

# ==========================================
# 4. INTERACTIVE RENDERER
# ==========================================
def view_real_stream(batch_index):
    state = stream_history[batch_index]
    plot_densoinn_topology(
        X_raw=state['X_raw'],
        nodes=state['nodes'],
        edges=state['edges'],
        macro_clusters=state['macro_clusters'],
        batch_name=state['batch_name']
    )

# Launch the interactive slider
interact(view_real_stream, batch_index=IntSlider(min=0, max=len(stream_history)-1, step=1, value=0, description='Time Step:'))

Loading 'complex9' data...
Processing 10 batches...
Stream processing complete! Ready for visualization.


interactive(children=(IntSlider(value=0, description='Time Step:', max=10), Output()), _dom_classes=('widget-i…

<function __main__.view_real_stream(batch_index)>